# Monte Carlo Valuation of Solar Assets

This notebook demonstrates stochastic valuation of a solar PV project:

1. **Stochastic price modeling** - Ornstein-Uhlenbeck process for mean-reverting electricity prices
2. **Monte Carlo simulation** - 10,000 scenarios for NPV distribution
3. **Risk metrics** - VaR, CVaR, probability of positive returns
4. **Sensitivity analysis** - Tornado charts for key drivers
5. **Scenario analysis** - Testing different market conditions

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.models.price_models import OUProcess, GBMProcess, SeasonalOUProcess
from src.models.solar_asset import SolarAsset
from src.models.valuation import MonteCarloValuation, quick_valuation
from src.models.risk import sensitivity_analysis, scenario_analysis

## 1. Stochastic Price Models

Electricity prices mean-revert to marginal generation costs, unlike stock prices which follow random walks. We use the **Ornstein-Uhlenbeck** process:

$$dX_t = \kappa(\theta - X_t)dt + \sigma dW_t$$

Where:
- $\theta$ = long-term mean price (£/MWh)
- $\kappa$ = speed of mean reversion
- $\sigma$ = volatility

In [ ]:
# Define price process parameters (calibrated to UK market)
price_process = OUProcess(
    theta=60.0,    # Long-term mean: £60/MWh
    kappa=0.5,     # Mean reversion half-life: ~1.4 years
    sigma=15.0,    # Volatility: £15/MWh
    x0=65.0,       # Current price: £65/MWh
)

# Simulate 100 price paths over 25 years
n_display_paths = 100
n_years = 25

paths = price_process.simulate(
    n_paths=n_display_paths,
    n_steps=n_years,
    dt=1.0,  # Annual steps
    seed=42,
)

print(f"Simulated {n_display_paths} price paths over {n_years} years")
print(f"Year 1 prices: mean={paths[:, 0].mean():.1f}, std={paths[:, 0].std():.1f}")
print(f"Year 25 prices: mean={paths[:, -1].mean():.1f}, std={paths[:, -1].std():.1f}")

In [ ]:
# Visualize price paths
fig = go.Figure()

# Plot individual paths (faded)
for i in range(min(50, n_display_paths)):
    fig.add_trace(go.Scatter(
        x=list(range(1, n_years + 1)),
        y=paths[i],
        mode='lines',
        line=dict(color='blue', width=0.5),
        opacity=0.2,
        showlegend=False,
    ))

# Plot mean path
fig.add_trace(go.Scatter(
    x=list(range(1, n_years + 1)),
    y=paths.mean(axis=0),
    mode='lines',
    line=dict(color='red', width=3),
    name='Mean Path',
))

# Plot long-term mean
fig.add_hline(y=price_process.theta, line_dash="dash", line_color="green",
              annotation_text=f"θ = £{price_process.theta}/MWh")

fig.update_layout(
    title="Simulated Electricity Price Paths (OU Process)",
    xaxis_title="Year",
    yaxis_title="Price (£/MWh)",
    yaxis_range=[0, 150],
)
fig.show()

## 2. Solar Asset Model

Define a 50MW solar farm with UK-typical parameters.

In [ ]:
# Define the solar asset
solar_farm = SolarAsset(
    capacity_mw=50.0,
    capacity_factor=0.11,          # UK average ~11%
    annual_degradation=0.005,      # 0.5%/year
    capex_per_mw=450_000,          # £450k/MW
    opex_fixed_per_mw=8_000,       # £8k/MW/year
    opex_variable_per_mwh=0.5,     # £0.5/MWh
    project_life_years=25,
    debt_fraction=0.7,
    cost_of_debt=0.05,
    cost_of_equity=0.10,
    tax_rate=0.25,
)

print(f"Project Parameters:")
print(f"  Capacity: {solar_farm.capacity_mw} MW")
print(f"  Total CAPEX: £{solar_farm.total_capex:,.0f}")
print(f"  Annual Fixed OPEX: £{solar_farm.annual_opex_fixed:,.0f}")
print(f"  WACC: {solar_farm.wacc:.2%}")
print(f"")
print(f"  Year 1 Generation: {solar_farm.annual_generation_mwh(0):,.0f} MWh")
print(f"  Year 25 Generation: {solar_farm.annual_generation_mwh(24):,.0f} MWh")

In [ ]:
# Visualize generation and degradation
years = np.arange(1, 26)
generation = solar_farm.generation_profile(25)

fig = px.bar(
    x=years,
    y=generation,
    title="Annual Generation Profile (with degradation)",
    labels={"x": "Year", "y": "Generation (MWh)"},
)
fig.update_layout(showlegend=False)
fig.show()

## 3. Monte Carlo Valuation

Run 10,000 simulations to build NPV distribution.

In [ ]:
# Set up Monte Carlo valuation
mc_valuation = MonteCarloValuation(
    asset=solar_farm,
    electricity_process=price_process,
    n_simulations=10_000,
    seed=42,
)

# Run valuation
result = mc_valuation.run(store_paths=True)

# Print summary
print(result.summary())

In [ ]:
# NPV Distribution Histogram
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=result.npv_samples / 1e6,  # Convert to millions
    nbinsx=100,
    name="NPV Distribution",
    marker_color='steelblue',
))

# Add vertical lines for key statistics
fig.add_vline(x=result.npv_mean / 1e6, line_dash="solid", line_color="red",
              annotation_text=f"Mean: £{result.npv_mean/1e6:.1f}M")
fig.add_vline(x=result.var_95 / 1e6, line_dash="dash", line_color="orange",
              annotation_text=f"VaR 95%: £{result.var_95/1e6:.1f}M")
fig.add_vline(x=0, line_dash="dot", line_color="black")

fig.update_layout(
    title=f"NPV Distribution (n={result.n_simulations:,} simulations)",
    xaxis_title="NPV (£ millions)",
    yaxis_title="Frequency",
)
fig.show()

In [ ]:
# NPV by percentile (fan chart style)
percentiles = [5, 10, 25, 50, 75, 90, 95]
pct_values = {p: np.percentile(result.npv_samples, p) for p in percentiles}

fig = go.Figure()

fig.add_trace(go.Bar(
    x=[f"{p}th" for p in percentiles],
    y=[pct_values[p] / 1e6 for p in percentiles],
    marker_color=['#d73027', '#fc8d59', '#fee090', '#ffffbf', '#e0f3f8', '#91bfdb', '#4575b4'],
))

fig.add_hline(y=0, line_dash="dash", line_color="black")

fig.update_layout(
    title="NPV Percentiles",
    xaxis_title="Percentile",
    yaxis_title="NPV (£ millions)",
)
fig.show()

In [ ]:
# IRR distribution (if available)
if result.irr_samples is not None:
    valid_irr = result.irr_samples[result.irr_samples != 0]
    
    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=valid_irr * 100,  # Convert to percentage
        nbinsx=50,
        marker_color='green',
    ))
    
    fig.add_vline(x=result.irr_mean * 100, line_dash="solid", line_color="red",
                  annotation_text=f"Mean IRR: {result.irr_mean:.1%}")
    fig.add_vline(x=solar_farm.wacc * 100, line_dash="dash", line_color="orange",
                  annotation_text=f"WACC: {solar_farm.wacc:.1%}")
    
    fig.update_layout(
        title="IRR Distribution",
        xaxis_title="IRR (%)",
        yaxis_title="Frequency",
    )
    fig.show()

## 4. Sensitivity Analysis

Test how NPV changes when key parameters vary.

In [ ]:
# Run sensitivity analysis
price_params = {
    "theta": 60.0,
    "kappa": 0.5,
    "sigma": 15.0,
    "x0": 65.0,
}

sensitivity_results = sensitivity_analysis(
    base_asset=solar_farm,
    base_price_params=price_params,
    n_simulations=2_000,  # Fewer sims for speed
    n_points=11,
)

print("Sensitivity Analysis Complete")
print("\nParameter Impact Ranges (NPV):")
for res in sensitivity_results:
    low, high = res.impact_range()
    print(f"  {res.parameter}: £{low/1e6:.1f}M to £{high/1e6:.1f}M")

In [ ]:
# Tornado chart
base_npv = result.npv_mean

# Calculate impact for each parameter
tornado_data = []
for res in sensitivity_results:
    low, high = res.impact_range()
    tornado_data.append({
        "parameter": res.parameter,
        "low_impact": (low - base_npv) / 1e6,
        "high_impact": (high - base_npv) / 1e6,
        "range": (high - low) / 1e6,
    })

# Sort by impact range
tornado_data.sort(key=lambda x: x["range"], reverse=True)

fig = go.Figure()

for i, d in enumerate(tornado_data):
    fig.add_trace(go.Bar(
        y=[d["parameter"]],
        x=[d["high_impact"] - d["low_impact"]],
        base=[d["low_impact"]],
        orientation='h',
        marker_color='steelblue',
        showlegend=False,
    ))

fig.add_vline(x=0, line_color="black")

fig.update_layout(
    title="Tornado Chart: NPV Sensitivity (±30% parameter variation)",
    xaxis_title="Change in NPV (£ millions)",
    yaxis_title="Parameter",
    height=400,
)
fig.show()

## 5. Scenario Analysis

Test different market and technology scenarios.

In [ ]:
# Run scenario analysis
scenarios = scenario_analysis(
    base_asset=solar_farm,
    base_price=60.0,
    n_simulations=10_000,
)

# Display results
print("Scenario Analysis Results")
print("=" * 70)
for s in scenarios:
    print(f"\n{s.name}")
    print(f"  {s.description}")
    print(f"  NPV: £{s.npv_mean/1e6:.1f}M ± £{s.npv_std/1e6:.1f}M")
    print(f"  P(NPV > 0): {s.probability_positive:.1%}")
    if s.irr_mean:
        print(f"  IRR: {s.irr_mean:.1%}")

In [ ]:
# Scenario comparison chart
fig = go.Figure()

names = [s.name for s in scenarios]
npvs = [s.npv_mean / 1e6 for s in scenarios]
stds = [s.npv_std / 1e6 for s in scenarios]

colors = ['steelblue' if npv > 0 else 'indianred' for npv in npvs]

fig.add_trace(go.Bar(
    x=names,
    y=npvs,
    error_y=dict(type='data', array=stds, visible=True),
    marker_color=colors,
))

fig.add_hline(y=0, line_dash="dash", line_color="black")

fig.update_layout(
    title="Scenario Analysis: Expected NPV (with 1σ error bars)",
    xaxis_title="Scenario",
    yaxis_title="NPV (£ millions)",
)
fig.show()

## 6. Key Takeaways

From this Monte Carlo analysis:

1. **Expected NPV** is positive, but there's significant uncertainty
2. **Electricity price** is the dominant risk factor (see tornado chart)
3. **VaR/CVaR** show downside risk that would matter for debt covenants
4. **Scenario analysis** reveals sensitivity to market structure changes

### Next Steps

- Calibrate OU parameters to historical price data
- Add correlation between prices and generation (negative: more sun = lower prices)
- Include PPA contracts for revenue stabilization
- Model carbon price uncertainty with GBM process